# RainGuard — Module 4: Feature Engineering
**Indian Rainfall Forecasting and Heavy Rainfall Risk Analysis System**

**Input:** `data/processed/rainfall_cleaned.csv` (Module 1) and `models/intensity_thresholds.json`.
**Output:** `data/processed/rainfall_features.csv` (the modelling table) and `models/feature_config.json` (feature lists and rules that Modules 5–8 reuse).

**The one idea behind this module.** We stand at the end of day *t* and want to forecast day *t + 1*. A feature is allowed **only if it could be computed at the end of day t**. Everything below follows from that rule.

```
row = (district, date t)
features : built from rainfall on day t and earlier days   -> known today
targets  : rainfall (mm) and intensity class on day t + 1  -> what we forecast
```

**What this notebook does**
1. Loads the data and builds a **complete calendar** per district (so a "previous day" is always a real previous calendar day).
2. Builds history features (lags, rolling sums/means/maxima, wet-day count), a running district average, calendar features and the encoded location.
3. Builds the three targets: next-day rainfall (mm), next-day intensity class, and next-day heavy-or-above.
4. **Proves there is no leakage** with four automatic tests.
5. Accounts for every row that is removed, checks the features against the target, and saves the table plus a feature configuration file.

**Not used as inputs (Module 1, Step 12):** `weekly_*`, `monthly_*`, `cumulative_*`, every `*_departure_pct` and every `*_category` column, and today's intensity *class*. Today's rainfall in **millimetres** is a legitimate input: at the end of day t it is a known number.

## Step 1 — Imports, paths and data

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
CLEAN_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_cleaned.csv")
THRESH_PATH = os.path.join(PROJECT_ROOT, "models", "intensity_thresholds.json")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
for p in (CLEAN_PATH, THRESH_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError("Run the Module 1 notebook first to create " + p)

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#b9b8b2", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "font.size": 10, "legend.frameon": False,
})

df = pd.read_csv(CLEAN_PATH, parse_dates=["date"])
with open(THRESH_PATH) as f:
    th = json.load(f)
INTENSITY_LABELS = th["labels"]
INTENSITY_BINS = [-np.inf if i == 0 else (np.inf if b is None else b) for i, b in enumerate(th["bins"])]
HEAVY_MM = 64.5
print("Cleaned data:", df.shape, "|", df["date"].min().date(), "->", df["date"].max().date())
print("Intensity classes:", INTENSITY_LABELS)

## Step 2 — Build a complete calendar for every district
**What we are doing:** a "previous day" must be the previous **calendar** day. If a district has a missing day, simply shifting rows by one position would silently use rainfall from two days earlier. So we create a full grid of every district × every date (728 × 73) and fill in the rainfall we have. Missing days become empty (`NaN`) instead of being skipped.

This is the same rule as Module 1's `days_since_prev_record`, applied in a cleaner way: after this step, `shift(1)` always means "exactly one calendar day earlier".

In [ ]:
ALL_DATES = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
keys = df[["state", "district", "state_enc", "district_enc"]].drop_duplicates()
assert len(keys) == 728

grid = keys.merge(pd.DataFrame({"date": ALL_DATES}), how="cross")
panel = (grid.merge(df[["state", "district", "date", "daily_actual", "daily_normal"]],
                    on=["state", "district", "date"], how="left")
             .sort_values(["state", "district", "date"]).reset_index(drop=True))

print("Calendar grid:", panel.shape, "=", keys.shape[0], "districts x", len(ALL_DATES), "dates")
gaps = panel[panel["daily_actual"].isna()]
print("Calendar slots with no record:", len(gaps))
display(gaps.groupby(["state", "district"]).size().to_frame("missing_days"))

**Findings.** Only **42 of 53,144** slots are empty, all from one district: **Nagapattinam (Tamil Nadu)**, the district flagged in Modules 1 and 2. Every other district has a complete daily record. We keep the gaps as `NaN`; no value is invented.

## Step 3 — Build the features
**What we are doing:** one function, `build_features`, creates every feature **per district** (`groupby(["state","district"])`) so one district's rain never leaks into another's. We keep it as a function on purpose: Step 5 runs it twice to prove there is no leakage, and the Flask app in Module 8 must compute the same features the same way.

| Feature | Meaning (all known at the end of day t) |
|---|---|
| `rain_today` | rainfall on day t (mm) |
| `rain_lag1`, `rain_lag2` | rainfall 1 and 2 days before t |
| `rain_sum3`, `rain_sum7` | total rainfall over the last 3 / 7 days (including t) |
| `rain_mean3`, `rain_mean7` | rolling average over the last 3 / 7 days |
| `rain_max7` | heaviest single day in the last 7 days |
| `wet_days7` | how many of the last 7 days had rain |
| `dist_mean_to_date` | the district's average daily rainfall from the first date up to t (a "how wet is this place" number that never looks ahead) |
| `normal_next` | IMD's long-term *normal* for day t + 1 (climatology: it is a published average, not computed from actual rain, and it is known in advance) |
| `year`, `month`, `day`, `day_of_week`, `day_of_year` | calendar features of day t |
| `state_enc`, `district_enc` | integer IDs from Module 1 (same encoders reused) |

**Rule for windows:** a rolling window is computed only when **all** its days exist (`min_periods = window`). A 7-day feature never silently averages 5 days.

**Honest note.** `rain_mean3 = rain_sum3 / 3` and `rain_mean7 = rain_sum7 / 7`, so each sum/mean pair carries identical information. Your brief lists both "previous N-day rainfall" and "rolling average", so we create both, but Module 5 uses only one of each pair (see Step 9).

In [ ]:
def build_features(p):
    # p: complete district x date grid with daily_actual / daily_normal. Returns a copy with features and the next-day target.
    p = p.copy()
    g = p.groupby(["state", "district"], sort=False)
    r = g["daily_actual"]

    def roll(fn, w):
        return r.transform(lambda s: getattr(s.rolling(w, min_periods=w), fn)())

    # --- rainfall history: only day t and earlier ---
    p["rain_today"] = p["daily_actual"]
    p["rain_lag1"] = r.shift(1)
    p["rain_lag2"] = r.shift(2)
    p["rain_sum3"], p["rain_sum7"] = roll("sum", 3), roll("sum", 7)
    p["rain_mean3"], p["rain_mean7"] = roll("mean", 3), roll("mean", 7)
    p["rain_max7"] = roll("max", 7)
    p["wet_days7"] = r.transform(lambda s: (s > 0).astype(float).where(s.notna()).rolling(7, min_periods=7).sum())
    p["dist_mean_to_date"] = r.transform(lambda s: s.expanding().mean())

    # --- climatology for the day we forecast (published long-term normal, not derived from actual rain) ---
    p["normal_next"] = g["daily_normal"].shift(-1)

    # --- calendar features of day t ---
    p["year"], p["month"], p["day"] = p["date"].dt.year, p["date"].dt.month, p["date"].dt.day
    p["day_of_week"], p["day_of_year"] = p["date"].dt.dayofweek, p["date"].dt.dayofyear

    # --- target: rainfall on the next calendar day of the same district ---
    p["target_date"] = p["date"] + pd.Timedelta(days=1)
    p["target_next_mm"] = r.shift(-1)
    return p

FEATURES_ALL = ["rain_today", "rain_lag1", "rain_lag2", "rain_sum3", "rain_sum7", "rain_mean3", "rain_mean7",
                "rain_max7", "wet_days7", "dist_mean_to_date", "normal_next",
                "year", "month", "day", "day_of_week", "day_of_year", "state_enc", "district_enc"]

feat = build_features(panel)
print("Feature table (before dropping incomplete rows):", feat.shape)
display(feat[["state", "district", "date", "rain_today", "rain_lag1", "rain_sum3", "rain_mean7", "rain_max7", "wet_days7",
              "dist_mean_to_date", "normal_next", "target_next_mm"]].iloc[5:12])

**What to expect.** The first rows of a district have empty (`NaN`) 7-day features because a full 7-day window needs seven days of history; the last date has no next day, so its target is empty. Both are removed in Step 6 and counted exactly.

## Step 4 — Targets
**What we are doing:** three targets, all describing day **t + 1**:
* `target_next_mm` — next-day rainfall in mm (regression, Module 5), created in Step 3 with a per-district shift of −1 calendar day.
* `target_next_class` / `target_next_code` — the next day's IMD intensity class, using the **same thresholds** saved in Module 1 (classification, Module 6).
* `target_next_heavy` — 1 if next-day rainfall is at least 64.5 mm (Heavy or above), else 0. We create it now because Modules 6–7 will report the headline "heavy-rain recall" on this combined group (EDA warned that Very Heavy and Extremely Heavy are too rare to judge separately).

Rows with an empty target are dropped in Step 6, so we do not label them here.

## Step 5 — Leakage audit (four automatic tests)
**Why:** leakage is the most common reason a rainfall model looks excellent and then fails. Instead of just claiming there is none, each test below fails loudly (`assert`) if something is wrong.

1. **Future-change test.** Replace all rainfall **after 1 October** with an absurd value (999 mm), rebuild the features, and check that every feature for dates up to 1 October is *exactly* unchanged. If any feature used tomorrow's or later data, it would change. (`normal_next` is excluded for the single row on 1 October itself, because by design it reads day t + 1's published normal; we check it for earlier dates.)
2. **Target-alignment test.** The target must equal the same district's actual rainfall on the next calendar day.
3. **Hand-check test.** Recompute some features for a chosen district and date directly from the raw cleaned file and compare.
4. **Forbidden-column test.** No feature name may contain `category`, `weekly`, `monthly`, `cumulative`, `departure`, `intensity` or `target`.

In [ ]:
# ---- Test 1: future-change test ----
CUT = pd.Timestamp("2026-10-01")
tampered = panel.copy()
after = tampered["date"] > CUT
tampered.loc[after, "daily_actual"] = 999.0
tampered.loc[after, "daily_normal"] += 500.0
feat_t = build_features(tampered)

past = feat["date"] <= CUT
same_cols = [c for c in FEATURES_ALL if c != "normal_next"]
for c in same_cols:
    assert np.allclose(feat.loc[past, c].fillna(-1), feat_t.loc[past, c].fillna(-1)), f"LEAK: {c} changed when the future changed"
before_cut = feat["date"] < CUT
assert np.allclose(feat.loc[before_cut, "normal_next"].fillna(-1), feat_t.loc[before_cut, "normal_next"].fillna(-1))
print(f"Test 1 PASSED: all {len(FEATURES_ALL)} features are unchanged for dates up to {CUT.date()} when the future is altered")

# ---- Test 2: target alignment ----
ok_rows = feat.dropna(subset=["target_next_mm"])
raw_next = df[["state", "district", "date", "daily_actual"]].rename(columns={"date": "target_date", "daily_actual": "raw_next"})
chk = ok_rows.merge(raw_next, on=["state", "district", "target_date"], how="left")
assert chk["raw_next"].notna().all() and np.allclose(chk["target_next_mm"], chk["raw_next"])
print(f"Test 2 PASSED: target_next_mm equals the next calendar day's actual rainfall for all {len(chk):,} rows")

# ---- Test 3: hand-check against the raw cleaned file ----
S, D, T = "ODISHA", "KORAPUT", pd.Timestamp("2026-09-24")
hist = df[(df["state"] == S) & (df["district"] == D)].set_index("date")["daily_actual"]
row = feat[(feat["state"] == S) & (feat["district"] == D) & (feat["date"] == T)].iloc[0]
manual = {"rain_today": hist[T], "rain_lag1": hist[T - pd.Timedelta(days=1)],
          "rain_sum3": hist[T - pd.Timedelta(days=2):T].sum(), "rain_mean7": hist[T - pd.Timedelta(days=6):T].mean(),
          "rain_max7": hist[T - pd.Timedelta(days=6):T].max(), "wet_days7": (hist[T - pd.Timedelta(days=6):T] > 0).sum(),
          "target_next_mm": hist[T + pd.Timedelta(days=1)]}
for k, v in manual.items():
    assert np.isclose(row[k], v), (k, row[k], v)
print(f"Test 3 PASSED: features for {D.title()} on {T.date()} match a manual calculation:",
      {k: round(float(v), 2) for k, v in manual.items()})

# ---- Test 4: forbidden columns ----
FORBIDDEN = ["category", "weekly", "monthly", "cumulative", "departure", "intensity", "target"]
bad = [c for c in FEATURES_ALL if any(w in c for w in FORBIDDEN)]
assert not bad, bad
print("Test 4 PASSED: no feature name contains", FORBIDDEN)

**What to expect.** All four lines print `PASSED`. If one fails, the message names the offending feature.

## Step 6 — Remove incomplete rows and account for every one
**What we are doing:** a row can be used for training only if **all** its features and its target exist. We remove the others and explain *every* removed row, so no data disappears silently.

In [ ]:
need = FEATURES_ALL + ["target_next_mm"]
incomplete = feat[need].isna().any(axis=1)

first_window = feat["date"] < ALL_DATES[6]          # first 6 dates: no full 7-day history
last_day = feat["date"] == ALL_DATES[-1]            # 30 Oct: no day t+1
reason = np.where(first_window, "first 6 dates (no 7-day history)",
          np.where(last_day, "last date (no next day to forecast)", "other gap (missing days nearby)"))
acc = pd.Series(reason[incomplete.values]).value_counts().to_frame("rows removed")
acc.loc["TOTAL removed"] = acc["rows removed"].sum()
acc.loc["rows kept"] = (~incomplete).sum()
display(acc)

model_df = feat.loc[~incomplete].copy()

# class targets (same IMD thresholds as Module 1)
model_df["target_next_class"] = pd.cut(model_df["target_next_mm"], bins=INTENSITY_BINS, labels=INTENSITY_LABELS)
model_df["target_next_code"] = model_df["target_next_class"].cat.codes
model_df["target_next_heavy"] = (model_df["target_next_mm"] >= HEAVY_MM).astype(int)
assert model_df["target_next_class"].notna().all() and (model_df["target_next_code"] >= 0).all()

print("\nDistricts in the modelling table:", model_df["district"].nunique(), "of 728")
left_out = sorted(set(keys["district"]) - set(model_df["district"]))
print("Districts with no usable rows:", left_out)
print("Origin dates (t):", model_df["date"].min().date(), "->", model_df["date"].max().date(),
      "| forecast dates (t+1):", model_df["target_date"].min().date(), "->", model_df["target_date"].max().date())

**Findings.**
* The table keeps **47,982 rows from 727 districts**: 727 districts × 66 usable origin dates (25 Aug – 29 Oct), so forecasts cover **26 Aug – 30 Oct**.
* Removed rows are fully explained: the first 6 dates of every district (no 7-day history), the last date (no next day), and a few rows around Nagapattinam's gaps.
* **Nagapattinam drops out completely.** Its 31 records are scattered across the period and its longest unbroken run is only 6 days, so it never has the 7 consecutive days a full window needs, and it cannot be used honestly. We document it as a limitation rather than fill the gaps.

## Step 7 — A worked example you can show in the viva
**What we are doing:** eight consecutive days for one district, so the shifts are visible. Read each row as *"standing at the end of this date, what do we know, and what happens tomorrow?"* — and notice that the `target_next_mm` on one row equals `rain_today` on the next row.

In [ ]:
ex = model_df[(model_df["district"] == "KORAPUT") & (model_df["date"] >= "2026-09-22") & (model_df["date"] <= "2026-09-29")]
display(ex[["date", "rain_today", "rain_lag1", "rain_lag2", "rain_sum3", "rain_mean7", "rain_max7", "wet_days7",
            "target_next_mm", "target_next_class"]].reset_index(drop=True))

**What to expect.** Each row's `rain_lag1` is the previous row's `rain_today`, and each row's `target_next_mm` is the next row's `rain_today`. This is the visible proof that features look backwards and the target looks exactly one day forward.

## Step 8 — Sanity checks: do the features relate to the target?
**What we are doing:** a quick Spearman correlation of every feature with `target_next_mm` (as in Module 3, Step 8). We are not selecting features from this; we are checking that the engineered table behaves as the statistics predicted, and that useless features are identified. `year` is constant, so it has no correlation and is left out of the chart.

In [ ]:
cand = [c for c in FEATURES_ALL if model_df[c].nunique() > 1]
sp = pd.Series({c: stats.spearmanr(model_df[c], model_df["target_next_mm"])[0] for c in cand}).sort_values()
display(sp.iloc[::-1].round(3).to_frame("Spearman with next-day rainfall"))
print("Constant features (no information):", [c for c in FEATURES_ALL if model_df[c].nunique() <= 1])

fig, ax = plt.subplots(figsize=(8, 5.6))
ax.barh(sp.index, sp.values, color=[ORANGE if v < 0 else BLUE for v in sp.values], height=0.65)
for i, v in enumerate(sp.values):
    ax.text(v + (0.008 if v >= 0 else -0.008), i, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=8.5, color=INK2)
ax.axvline(0, color=INK2, linewidth=0.8)
ax.set_xlim(min(sp.min() * 1.4, -0.1), sp.max() * 1.18)
ax.set(xlabel="Spearman correlation with next-day rainfall", title="Engineered features vs tomorrow's rainfall")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "m4_01_feature_correlation.png"), dpi=150); plt.show()

print("\nTarget classes in the modelling table:")
cls = model_df["target_next_class"].value_counts().reindex(INTENSITY_LABELS)
display(pd.DataFrame({"rows": cls, "percent": (cls / len(model_df) * 100).round(3)}))
print("Heavy-or-above next-day rows:", int(model_df["target_next_heavy"].sum()),
      f"({model_df['target_next_heavy'].mean() * 100:.2f}%)")

**Findings.**
* The ranking matches Module 3: `rain_today` (0.57) and the 3-day measures (0.51) lead, then `wet_days7` (0.49), the 7-day measures (0.43), `normal_next` (0.42) and `rain_max7` (0.39). `day_of_year` and `month` are negative (about −0.34) because rain declines through the season.
* **Useless or redundant features:** `year` is constant; `day` (0.00) and `day_of_week` (0.03) carry essentially no signal; `month` repeats what `day_of_year` says; and `state_enc` (0.01) / `district_enc` (0.04) are weak *as raw integers* (alphabetical labels, see Module 3, Step 8).
* The new `wet_days7` (0.49) is a strong addition: how many of the last 7 days were wet says more than the last-7-day maximum.
* Next-day classes keep the severe imbalance: **75.5%** No/Very Light, 16.7% Light, 7.2% Moderate, and only **247** Heavy, **48** Very Heavy and **6** Extremely Heavy rows. The heavy-or-above group has **301 rows (0.63%)**. Module 6 must handle this imbalance; the 6 Extremely Heavy rows are far too few to evaluate separately.

## Step 9 — Feature groups, saving, and what Module 5 will use
**What we are doing:** we do *not* pick the final features by looking at test results. We define **named groups** with a written reason, so Module 5 can test them in a fair, documented comparison (an "ablation"), and we save everything the later modules need:

| Group | Features | Reason |
|---|---|---|
| `history` | `rain_today`, `rain_lag1`, `rain_lag2`, `rain_mean3`, `rain_mean7`, `rain_max7`, `wet_days7` | Core signal (Module 3, Step 4). Sums are dropped because they equal mean × window |
| `season` | `day_of_year` | Smoother than `month`; `year` constant |
| `location_numeric` | `dist_mean_to_date` | A meaningful "how wet is this place" number that never looks ahead |
| `location_codes` | `state_enc`, `district_enc` | Tested as an extra; fine for tree models, meaningless as numbers for linear ones |
| `climatology` | `normal_next` | Published average, not derived from actual rain: tested with and without |

**Excluded and why:** `year` (constant), `month` (duplicate of `day_of_year`), `day` and `day_of_week` (no signal), `rain_sum3` / `rain_sum7` (collinear with the means). They stay in the saved table because your brief lists them, but they are not in any group.

In [ ]:
FEATURE_GROUPS = {
    "history": ["rain_today", "rain_lag1", "rain_lag2", "rain_mean3", "rain_mean7", "rain_max7", "wet_days7"],
    "season": ["day_of_year"],
    "location_numeric": ["dist_mean_to_date"],
    "location_codes": ["state_enc", "district_enc"],
    "climatology": ["normal_next"],
}
EXCLUDED = {"year": "constant (single year)", "month": "duplicates day_of_year", "day": "no signal (Spearman 0.00)",
            "day_of_week": "no signal (Spearman 0.03)", "rain_sum3": "= rain_mean3 x 3 (collinear)",
            "rain_sum7": "= rain_mean7 x 7 (collinear)"}

TARGET_COLS = ["target_next_mm", "target_next_class", "target_next_code", "target_next_heavy"]
ID_COLS = ["state", "district", "date", "target_date"]
out = model_df[ID_COLS + FEATURES_ALL + TARGET_COLS].sort_values(["state", "district", "date"]).reset_index(drop=True)

assert out[FEATURES_ALL + ["target_next_mm"]].notna().all().all()
assert not out.duplicated(subset=["state", "district", "date"]).any()
assert (out["target_date"] - out["date"]).dt.days.eq(1).all()
used = {c for grp_ in FEATURE_GROUPS.values() for c in grp_}
assert used.isdisjoint(EXCLUDED) and used | set(EXCLUDED) <= set(FEATURES_ALL)

OUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_features.csv")
out.to_csv(OUT_PATH, index=False)

config = {
    "rule": "features use only rainfall on day t and earlier; targets describe day t+1; windows need all days present",
    "windows": {"history_days": 7, "min_periods": "full window"},
    "features_all": FEATURES_ALL, "feature_groups": FEATURE_GROUPS, "excluded": EXCLUDED,
    "targets": {"regression": "target_next_mm", "classification": "target_next_class",
                "classification_code": "target_next_code", "heavy_binary": "target_next_heavy"},
    "heavy_threshold_mm": HEAVY_MM, "intensity_labels": INTENSITY_LABELS,
    "origin_dates": [str(out["date"].min().date()), str(out["date"].max().date())],
    "rows": int(len(out)), "districts": int(out["district"].nunique()),
}
with open(os.path.join(PROJECT_ROOT, "models", "feature_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Saved:", OUT_PATH)
print("Shape:", out.shape, "| districts:", out["district"].nunique())
print("Saved:", os.path.join(PROJECT_ROOT, "models", "feature_config.json"))
display(out.head())

## Module 4 summary
| Item | Result |
|---|---|
| Calendar grid | 728 districts × 73 dates; only Nagapattinam has gaps (42 empty slots) |
| Features created | 18: rainfall history (lags, sums, means, max, wet days), running district mean, next-day normal, calendar, location codes |
| Targets | next-day mm, next-day IMD class, next-day heavy-or-above (≥ 64.5 mm) |
| Leakage audit | 4 automatic tests passed (future-change, target alignment, hand-check, forbidden columns) |
| Modelling table | 47,982 rows, 727 districts, forecasts for 26 Aug – 30 Oct; 301 heavy-or-above targets (0.63%) |
| Dropped | first 6 dates, last date, Nagapattinam (all documented) |
| Feature groups for Module 5 | history, season, location_numeric, location_codes, climatology |

**Limitations to report.** Only 73 days, so features cannot capture a full seasonal cycle; the first 6 dates are lost to the 7-day window; Nagapattinam is excluded; `dist_mean_to_date` is noisy on the earliest dates; and the 15 duplicated-series districts (Module 2) still enter the table. When the Flask app is built, a user will supply recent rainfall history, so the feature function from Step 3 will be moved into a shared `.py` file and reused unchanged.

### Viva questions for Module 4
**Q1. How did you make sure the features do not leak the future?**
Features use only day t and earlier, grouped per district, and a test rebuilds them after replacing all later rainfall with 999 mm: for earlier dates every feature is exactly unchanged.

**Q2. Why build a full calendar grid before shifting?**
Because `shift(1)` on a table with missing days would reach back two days instead of one. The grid makes every shift a true calendar-day shift and turns gaps into explicit missing values.

**Q3. Why drop rows instead of filling missing history?**
Filling would invent rainfall. Losing 6 early dates per district and one district is a documented, honest cost.

**Q4. Why are `rain_sum7` and `rain_mean7` not both used in the models?**
The mean is the sum divided by 7, so they carry identical information. Keeping both adds collinearity and no new signal.

**Q5. Why is `normal_next` allowed but `monthly_actual` is not?**
`normal_next` is a published long-term average for tomorrow, known in advance and not computed from this year's rainfall. `monthly_actual` is an overlapping total that already contains today's rain and cannot be proven free of leakage.

*Next: Module 5 — Regression models (chronological split, baselines, Linear Regression, Random Forest, Gradient Boosting).*